### Word Document Processing

---

### 💡 Interview & Learning Notes

**Key Interview Questions:**
1. *Why use Unstructured over simpler loaders like Docx2txtLoader?* 
   - `Unstructured` parses documents into logical elements (e.g., Title, NarrativeText, Table). This means you can split text semantically rather than arbitrarily by character count.
2. *What is "mode='elements'" in Unstructured loaders?* 
   - Instead of returning one giant string per page, `mode="elements"` returns a separate LangChain Document object for each structural element (like a paragraph or table) identified in the file.

**Learning Takeaways:**
- For simple text extraction from Word files, `Docx2txtLoader` is fast and lightweight.
- For complex layouts containing tables and nested lists, `UnstructuredWordDocumentLoader` is superior, though it requires more heavy dependencies (like `libmagic` and `poppler`).

In [ ]:
"""
PURPOSE:
Load DOCX files for RAG pipelines.

INSIGHTS:
Standalone loaders are more future stable.
"""

# Why are standalone integrations preferred over langchain_community?
# Answer: To allow for faster updates and isolated dependency trees, especially for heavy integrations like Unstructured.

# Component Explanations:
# 1. Docx2txtLoader:
#    - Purpose: Extracts pure text from `.docx` files.
#    - Internally uses: The `docx2txt` python package.
#    - Why Used: Very fast, no external system dependencies, good for plain text documents.
# 2. UnstructuredWordDocumentLoader:
#    - Purpose: Extracts text while preserving document structure (titles, tables, text blocks).
#    - Internally uses: The `unstructured` library.
#    - Why Used: Essential when document layout and element categorization matter for retrieval.

'\nPURPOSE:\nLoad DOCX files for RAG pipelines.\n\nINSIGHTS:\nStandalone loaders are more future stable.\n'

In [ ]:

from langchain_community.document_loaders import Docx2txtLoader, UnstructuredWordDocumentLoader

C:\Users\DELL\AppData\Local\Temp\ipykernel_9500\3684629006.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader, UnstructuredWordDocumentLoader


In [ ]:
## Method 1: Using Docx2txtLoader
print("1️⃣ Using Docx2txtLoader")

try:
    docx_loader=Docx2txtLoader("data/word_files/proposal.docx")
    docs=docx_loader.load()

    print(f"✅ Loaded {len(docs)} document(s)")
    print(f"Content preview: {docs[0].page_content[:200]}...")
    print(f"Metadata: {docs[0].metadata}")

except Exception as e:
    print(f"Error: {e}")

1️⃣ Using Docx2txtLoader
✅ Loaded 1 document(s)
Content preview: Project Proposal: RAG Implementation

Executive Summary

This proposal outlines the implementation of a Retrieval-Augmented Generation system for our organization.

Objectives

Key objectives include:...
Metadata: {'source': 'data/word_files/proposal.docx'}


In [ ]:
from unstructured.partition.docx import partition_docx

print("START")

elements = partition_docx(
    filename="data/word_files/proposal.docx"
)

print("DONE")
print(len(elements))

START
DONE
20


In [ ]:
"""
PURPOSE:
Load DOCX files into structured document elements.

INSIGHTS:
Element-based parsing improves semantic retrieval.
"""

## Method 2
print("\n2️⃣ Using UnstructuredWordDocumentLoader")

try:
    unstructured_loader=UnstructuredWordDocumentLoader(
        "data/word_files/proposal.docx",
        mode="elements",
         strategy="fast"
    )

    # Why is element-based document parsing useful in RAG?
    # Answer: It allows you to filter or chunk by semantic type (e.g., only embed NarrativeText and ignore Headers/Footers), drastically improving retrieval quality.
    unstructured_docs=unstructured_loader.load()

    print(f"✅ Loaded {len(unstructured_docs)} elements")

    for i,doc in enumerate(unstructured_docs[:3]):
        print(f"\nElement {i+1}:")
        print(f"Type: {doc.metadata.get('category','unknown')}")
        print(f"Content: {doc.page_content[:100]}...")

except Exception as e:
    print(e)


2️⃣ Using UnstructuredWordDocumentLoader


In [ ]:
unstructured_docs

[Document(metadata={'source': 'data/word_files/proposal.docx', 'category_depth': 0, 'file_directory': 'data/word_files', 'filename': 'proposal.docx', 'last_modified': '2025-06-28T15:37:11', 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'Title', 'element_id': 'bb0410bfd160ef866f8d4357b0949db2'}, page_content='Project Proposal: RAG Implementation'),
 Document(metadata={'source': 'data/word_files/proposal.docx', 'category_depth': 0, 'file_directory': 'data/word_files', 'filename': 'proposal.docx', 'last_modified': '2025-06-28T15:37:11', 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'Title', 'element_id': 'c0f844859abf08d9506856b3aed4a719'}, page_content='Executive Summary'),
 Document(metadata={'source': 'data/word_files/proposal.docx', 'category_depth': 0, 'file_directory': 'data/word_files', 'filename': 'proposal.docx', 'last_modified': '2

### 🚀 Best Practices for Word Document Ingestion

1. **Cost Efficiency & Token Optimization**:
   - If using `Unstructured`, use the metadata to strip out `Header`, `Footer`, and `PageBreak` elements before embedding. This saves tokens and removes confusing noise for the LLM.

2. **Time Optimization**:
   - Use `Docx2txtLoader` if the document is primarily text without complex tables. It is significantly faster than `Unstructured`.
   - When using `Unstructured`, utilize `strategy="fast"` for speed when OCR is not required, and `strategy="hi_res"` only when you need bounding boxes or image parsing.

3. **Data Quality**:
   - Word documents often contain invisible revision history or comments. If security is a concern, ensure the `.docx` is "cleaned" of metadata before ingestion.